In [41]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [42]:
csv_files = [
    "../DATA/rtt_summary.csv",
]

df = pd.read_csv(csv_files[0])
df

C:\Users\crazy\AppData\Local\Temp\ipykernel_19796\2396639505.py:5: DtypeWarning: Columns (0: address_low_frac, 1: street_postdir, 2: receipt_num) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(csv_files[0])


,document_id,document_type,display_date,street_address,zip_code,grantors,grantees,cash_consideration,other_consideration,total_consideration,...,receipt_num,adjusted_assessed_value,adjusted_cash_consideration,adjusted_fair_market_value,adjusted_local_tax_amount,adjusted_other_consideration,adjusted_state_tax_amount,adjusted_total_consideration,ward,shape
0,18,MORTGAGE,2024-06-26 18:22:03+00,1928 S LAMBERT ST,191452825.0,DESIMONE JAMES,FIRSTRUST BANK,0.0,NaN,0.0,...,NaN,NaN,0.0,NaN,NaN,NaN,NaN,0.0,48.0,SRID=2272;POINT(2689096.18088191 226962.367925...
1,2999998,DEED,1986-12-10 10:00:00+00,109 PLEASANT ST,191192145.0,CITY OF PHILADELPHIA,REDEVELOPMENT AUTHORITY OF THE CITY OF PHILADE...,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,22.0,SRID=2272;POINT(2686409.79091299 273520.347320...
2,2999998,DEED,1986-12-10 10:00:00+00,1108 S 27TH ST,191463819.0,CITY OF PHILADELPHIA,REDEVELOPMENT AUTHORITY OF THE CITY OF PHILADE...,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,36.0,SRID=2272;POINT(2686664.17783195 231153.751457...
3,2999998,DEED,1986-12-10 10:00:00+00,124 DIAMOND ST,191221721.0,CITY OF PHILADELPHIA,REDEVELOPMENT AUTHORITY OF THE CITY OF PHILADE...,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,18.0,SRID=2272;POINT(2701421.62476646 247002.336556...
4,2999998,DEED,1986-12-10 10:00:00+00,145 W LEHIGH AVE,191333836.0,CITY OF PHILADELPHIA,REDEVELOPMENT AUTHORITY OF THE CITY OF PHILADE...,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,7.0,SRID=2272;POINT(2701807.30662948 250569.670997...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5171814,53722566,SATISFACTION OF MORTGAGE,2020-09-25 23:00:50+00,12665 DUNKS FERRY RD,191541424.0,MORTGAGE ELECTRONIC REGISTRATION SYSTEMS INC;F...,BURHIMER MATTHEW;BURKHIMER MATTHEW,0.0,NaN,0.0,...,NaN,NaN,0.0,NaN,NaN,NaN,NaN,0.0,66.0,SRID=2272;POINT(2745585.01441736 292152.54526894)
5171815,53722567,DEED,2020-09-09 08:00:00+00,768 N LECOUNT ST,191302418.0,SANTROCK ROBERTA,DIMARCO DAVID;FAGAN KAITLIN,427000.0,0.0,427000.0,...,NaN,0.0,213500.0,0.0,6998.53,0.0,2135.0,213500.0,15.0,SRID=2272;POINT(2688497.15291486 242272.047330...
5171816,53722567,DEED,2020-09-09 08:00:00+00,768 N LECOUNT ST,191302418.0,SANTROCK ROBERTA,DIMARCO DAVID;FAGAN KAITLIN,427000.0,0.0,427000.0,...,NaN,0.0,213500.0,0.0,6998.53,0.0,2135.0,213500.0,15.0,SRID=2272;POINT(2688497.15291486 242272.047330...
5171817,53722568,MORTGAGE,2020-09-25 23:01:20+00,768 N LECOUNT ST,191302418.0,DIMARCO DAVID;FAGAN KAITLIN,MORTGAGE ELECTRONIC REGISTRATION SYSTEMS INC;G...,0.0,NaN,0.0,...,NaN,NaN,0.0,NaN,NaN,NaN,NaN,0.0,15.0,SRID=2272;POINT(2688497.15291486 242272.047330...


In [43]:
START_DATE = pd.Timestamp("1999-01-01")
END_DATE = pd.Timestamp("2016-12-31")
ZIP_PREFIX = "19147"
PRICE_COL = "adjusted_cash_consideration"
PRICE_FLOOR = 10_000 
SD_CUTOFF = 3

steps = []
def log_step(name, before, after):
    steps.append({"step": name, "records_before": before,
                  "records_removed": before - after, "records_remaining": after})
    print(f"{name:<52} removed {before - after:>10,}   remaining {after:>10,}")

clean = df
log_step("Start: all records", len(clean), len(clean))

Start: all records                                   removed          0   remaining  5,171,819


In [44]:
# Keep only DEED
n = len(clean)
clean = clean[clean["document_type"].astype(str).str.strip().str.upper() == "DEED"].copy()
log_step("Keep document_type == DEED", n, len(clean))

# Parse document_date; impossible / unparseable dates become NaT and are dropped
n = len(clean)
clean["document_date"] = pd.to_datetime(clean["document_date"].astype(str).str[:10],
                                        format="%Y-%m-%d", errors="coerce")
clean = clean[clean["document_date"].notna()]
log_step("Remove impossible / unparseable dates", n, len(clean))

# date resrtictions 
n = len(clean)
clean = clean[clean["document_date"].between(START_DATE, END_DATE)]
log_step("Keep dates 1999-01-01 to 2016-12-31", n, len(clean))

# ZIP starts with 19147 
n = len(clean)
zip_str = clean["zip_code"].astype(str).str.strip().str.replace(r"\.0$", "", regex=True)
clean = clean[zip_str.str.startswith(ZIP_PREFIX)].copy()
clean["zip_code"] = zip_str[clean.index].str[:5]
log_step("Keep zip_code starting with 19147", n, len(clean))

Keep document_type == DEED                           removed  4,066,107   remaining  1,105,712
Remove impossible / unparseable dates                removed     35,323   remaining  1,070,389
Keep dates 1999-01-01 to 2016-12-31                  removed    378,518   remaining    691,871
Keep zip_code starting with 19147                    removed    666,172   remaining     25,699


In [45]:
# Remove records with no price
n = len(clean)
clean[PRICE_COL] = pd.to_numeric(clean[PRICE_COL], errors="coerce")
clean = clean[clean[PRICE_COL].notna()]
log_step("Remove records with no price", n, len(clean))


Remove records with no price                         removed        690   remaining     25,009


In [46]:
# remove nominal transfers
n = len(clean)
clean = clean[clean[PRICE_COL] >= PRICE_FLOOR]
log_step(f"Remove prices below floor (${PRICE_FLOOR:,})", n, len(clean))

# Remove same-day groups sharing one identical price (bulk / multi-property sales)
n = len(clean)
group_size = clean.groupby(["document_date", PRICE_COL])[PRICE_COL].transform("size")
clean = clean[group_size == 1]
log_step("Remove same-day identical-price groups", n, len(clean))

# Drop sales more than 3 SD from their year's mean log price
n = len(clean)
clean = clean.copy()
clean["year"] = clean["document_date"].dt.year
log_price = np.log(clean[PRICE_COL])
z = (log_price - log_price.groupby(clean["year"]).transform("mean")) / log_price.groupby(clean["year"]).transform("std")
clean = clean[z.abs().le(SD_CUTOFF) | z.isna()]
log_step(f"Remove sales > {SD_CUTOFF} SD from year mean log price", n, len(clean))

steps_df = pd.DataFrame(steps)
steps_df

Remove prices below floor ($10,000)                  removed      6,981   remaining     18,028
Remove same-day identical-price groups               removed      3,957   remaining     14,071
Remove sales > 3 SD from year mean log price         removed        171   remaining     13,900


,step,records_before,records_removed,records_remaining
0,Start: all records,5171819,0,5171819
1,Keep document_type == DEED,5171819,4066107,1105712
2,Remove impossible / unparseable dates,1105712,35323,1070389
3,Keep dates 1999-01-01 to 2016-12-31,1070389,378518,691871
4,Keep zip_code starting with 19147,691871,666172,25699
5,Remove records with no price,25699,690,25009
6,"Remove prices below floor ($10,000)",25009,6981,18028
7,Remove same-day identical-price groups,18028,3957,14071
8,Remove sales > 3 SD from year mean log price,14071,171,13900


In [47]:
# Month-year variable, monthly median price and sale count, interpolate empty months
clean["month_year"] = clean["document_date"].dt.to_period("M")

monthly = clean.groupby("month_year")[PRICE_COL].agg(median_price="median", sale_count="size")
monthly = monthly.reindex(pd.period_range(START_DATE, END_DATE, freq="M"))
monthly.index.name = "month_year"
monthly["sale_count"] = monthly["sale_count"].fillna(0).astype(int)
monthly["interpolated"] = monthly["median_price"].isna()
monthly["median_price"] = monthly["median_price"].interpolate(method="linear", limit_direction="both")

print(f"{len(monthly)} months; {monthly['interpolated'].sum()} had no sales and were interpolated")
monthly

216 months; 1 had no sales and were interpolated


,median_price,sale_count,interpolated
month_year,,,
1999-01,500000.0,1,False
1999-02,134250.0,2,False
1999-03,122725.0,0,True
1999-04,111200.0,2,False
1999-05,267500.0,2,False
...,...,...,...
2016-08,400000.0,75,False
2016-09,325000.0,63,False
2016-10,380000.0,62,False


In [48]:
# Save cleaned outputs next to the raw data, dropping unused columns
keep_cols = [
    "document_id",
    "document_date",
    "street_address",
    "zip_code",
    "adjusted_cash_consideration",
    "year",
    "month_year",
]
clean = clean[keep_cols]
clean.head()

clean_out = clean.copy()
clean_out["document_date"] = clean_out["document_date"].dt.date
clean_out["month_year"] = clean_out["month_year"].astype(str)
clean_out.to_csv("../DATA/rtt_19147_clean.csv", index=False)

monthly_out = monthly.reset_index()
monthly_out["month_year"] = monthly_out["month_year"].astype(str)
monthly_out.to_csv("../DATA/rtt_19147_monthly.csv", index=False)


steps_df.to_csv("../DATA/cleaning_steps_log.csv", index=False)
print("Saved rtt_19147_clean.csv, rtt_19147_monthly.csv, cleaning_steps_log.csv to ../DATA/")

Saved rtt_19147_clean.csv, rtt_19147_monthly.csv, cleaning_steps_log.csv to ../DATA/


In [49]:
clean = clean.sort_values(["document_date", "document_id"]).reset_index(drop=True)
clean

,document_id,document_date,street_address,zip_code,adjusted_cash_consideration,year,month_year
0,50829683,1999-01-07,600-36 WASHINGTON AVE UNIT 18U,19147,500000.0,1999,1999-01
1,49025851,1999-02-04,611 S 7TH ST,19147,218500.0,1999,1999-02
2,50021821,1999-02-26,1122 S 10TH ST,19147,50000.0,1999,1999-02
3,50009558,1999-04-16,613 S 11TH ST,19147,26000.0,1999,1999-04
4,50315017,1999-04-30,134 KENILWORTH ST,19147,196400.0,1999,1999-04
...,...,...,...,...,...,...,...
13895,53157774,2016-12-29,235 CATHARINE ST,19147,1090000.0,2016,2016-12
13896,53158245,2016-12-29,739 CHRISTIAN ST,19147,640665.0,2016,2016-12
13897,53164218,2016-12-29,235 GERRITT ST,19147,215000.0,2016,2016-12
13898,53164361,2016-12-29,628 LEAGUE ST,19147,200000.0,2016,2016-12
